# Part 6: Market Timing (A-share Timing Example)

**Overview**
- Market timing with macro/market indicators
- Decision rules and evaluation

**Workflow**
1. Build timing indicators
2. Define signal thresholds
3. Evaluate timing strategy performance



In [1]:
import pandas as pd
import numpy as np
from scipy.stats import percentileofscore

# 读取 .pkl 文件
data = pd.read_pickle('Example.pkl')
grouped_df = data.groupby('S_INFO_WINDCODE')

In [2]:
data

,S_INFO_WINDCODE,TRADE_DT,CRNCY_CODE,S_VAL_MV,S_DQ_MV,S_PQ_HIGH_52W_,S_PQ_LOW_52W_,S_VAL_PE,S_VAL_PB_NEW,S_VAL_PE_TTM,...,NET_PROFIT_PARENT_COMP_LYR,NET_ASSETS_TODAY,NET_CASH_FLOWS_OPER_ACT_TTM,NET_CASH_FLOWS_OPER_ACT_LYR,OPER_REV_TTM,OPER_REV_LYR,NET_INCR_CASH_CASH_EQU_TTM,NET_INCR_CASH_CASH_EQU_LYR,UP_DOWN_LIMIT_STATUS,LOWEST_HIGHEST_STATUS
13352774,603917.SH,20180320,CNY,2.731680e+05,6.829200e+04,32.99,20.48,34.8608,3.4073,31.5543,...,7.835970e+07,8.017039e+08,5.915746e+07,8.611410e+07,5.183946e+08,4.563433e+08,3.238071e+07,1.825670e+06,0.0,0.0
13352775,603917.SH,20180321,CNY,2.674560e+05,6.686400e+04,32.99,20.48,34.1318,3.3361,30.8945,...,7.835970e+07,8.017039e+08,5.915746e+07,8.611410e+07,5.183946e+08,4.563433e+08,3.238071e+07,1.825670e+06,0.0,0.0
13352776,603917.SH,20180322,CNY,2.786560e+05,6.966400e+04,32.99,20.48,35.5611,3.4758,32.1883,...,7.835970e+07,8.017039e+08,5.915746e+07,8.611410e+07,5.183946e+08,4.563433e+08,3.238071e+07,1.825670e+06,0.0,0.0
13352777,603917.SH,20180323,CNY,2.507680e+05,6.269200e+04,32.99,20.48,32.0022,3.1279,28.9668,...,7.835970e+07,8.017039e+08,5.915746e+07,8.611410e+07,5.183946e+08,4.563433e+08,3.238071e+07,1.825670e+06,-1.0,0.0
13352778,603917.SH,20180324,CNY,2.507680e+05,6.269200e+04,32.99,20.48,32.0022,3.1279,28.9668,...,7.835970e+07,8.017039e+08,5.915746e+07,8.611410e+07,5.183946e+08,4.563433e+08,3.238071e+07,1.825670e+06,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20923590,301107.SZ,20221128,CNY,2.119058e+05,5.303419e+04,60.03,26.89,25.8046,2.3701,25.9588,...,8.211940e+07,8.940620e+08,8.201421e+07,4.222326e+07,6.255461e+08,6.550237e+08,4.228356e+08,1.838927e+07,0.0,0.0
20923591,688556.SH,20221128,CNY,1.885610e+06,1.405111e+06,106.33,51.12,109.1891,11.9957,38.5363,...,1.726921e+08,1.571905e+09,2.176301e+07,7.649657e+07,2.784220e+09,1.566597e+09,1.547781e+08,1.269967e+08,0.0,0.0
20923592,605128.SH,20221128,CNY,4.273600e+05,2.228148e+05,66.47,19.93,60.6347,4.0487,74.2249,...,7.048109e+07,1.055558e+09,3.742761e+07,1.001508e+08,1.013144e+09,8.265074e+08,-1.670771e+08,4.306552e+07,0.0,0.0
20923593,301195.SZ,20221128,CNY,6.988188e+05,1.610550e+05,90.18,62.00,47.4046,3.3720,37.5596,...,1.474158e+08,2.072410e+09,5.420511e+07,6.210872e+07,7.186100e+08,5.781697e+08,2.109732e+08,1.815459e+07,0.0,0.0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7440297 entries, 13352774 to 20923594
Data columns (total 36 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   S_INFO_WINDCODE              object 
 1   TRADE_DT                     object 
 2   CRNCY_CODE                   object 
 3   S_VAL_MV                     float64
 4   S_DQ_MV                      float64
 5   S_PQ_HIGH_52W_               float64
 6   S_PQ_LOW_52W_                float64
 7   S_VAL_PE                     float64
 8   S_VAL_PB_NEW                 float64
 9   S_VAL_PE_TTM                 float64
 10  S_VAL_PCF_OCF                float64
 11  S_VAL_PCF_OCFTTM             float64
 12  S_VAL_PCF_NCF                float64
 13  S_VAL_PCF_NCFTTM             float64
 14  S_VAL_PS                     float64
 15  S_VAL_PS_TTM                 float64
 16  S_DQ_TURN                    float64
 17  S_DQ_FREETURNOVER            float64
 18  TOT_SHR_TODAY                float64
 1

In [4]:
data.columns

Index(['S_INFO_WINDCODE', 'TRADE_DT', 'CRNCY_CODE', 'S_VAL_MV', 'S_DQ_MV',
       'S_PQ_HIGH_52W_', 'S_PQ_LOW_52W_', 'S_VAL_PE', 'S_VAL_PB_NEW',
       'S_VAL_PE_TTM', 'S_VAL_PCF_OCF', 'S_VAL_PCF_OCFTTM', 'S_VAL_PCF_NCF',
       'S_VAL_PCF_NCFTTM', 'S_VAL_PS', 'S_VAL_PS_TTM', 'S_DQ_TURN',
       'S_DQ_FREETURNOVER', 'TOT_SHR_TODAY', 'FLOAT_A_SHR_TODAY',
       'S_DQ_CLOSE_TODAY', 'S_PRICE_DIV_DPS', 'S_PQ_ADJHIGH_52W',
       'S_PQ_ADJLOW_52W', 'FREE_SHARES_TODAY', 'NET_PROFIT_PARENT_COMP_TTM',
       'NET_PROFIT_PARENT_COMP_LYR', 'NET_ASSETS_TODAY',
       'NET_CASH_FLOWS_OPER_ACT_TTM', 'NET_CASH_FLOWS_OPER_ACT_LYR',
       'OPER_REV_TTM', 'OPER_REV_LYR', 'NET_INCR_CASH_CASH_EQU_TTM',
       'NET_INCR_CASH_CASH_EQU_LYR', 'UP_DOWN_LIMIT_STATUS',
       'LOWEST_HIGHEST_STATUS'],
      dtype='object')

In [5]:
riskfree = pd.read_excel('中国_10年期国债收益率.xlsx', names=['date', 'interest'])
riskfree = riskfree.drop(0).reset_index()
riskfree['date'] = pd.to_datetime(riskfree['date'])
riskfree['date'] = riskfree['date'].dt.strftime('%Y%m%d')

closeprice = pd.read_pickle('IndexQuote_ClosePrice.txt')

closeprice_500 = closeprice['000905.SH'].loc['20181030':'20221129']

contents_000905  = pd.read_pickle('IndexComponent_000905.txt')
contents_000905 = contents_000905.sort_index()

trade_date_list_000905 = list(contents_000905.index)

start_date = '20110101'
end_date = '20230531'

In [6]:
riskfree

,index,date,interest
0,1,20150104,3.6405
1,2,20150105,3.6406
2,3,20150106,3.6456
3,4,20150107,3.6505
4,5,20150108,3.6457
...,...,...,...
2106,2107,20230613,2.6583
2107,2108,20230614,2.6353
2108,2109,20230615,2.6654
2109,2110,20230616,2.7033


In [7]:
contents_000905

,000005.SZ,000006.SZ,000008.SZ,000009.SZ,000012.SZ,000016.SZ,000021.SZ,000025.SZ,000027.SZ,000028.SZ,...,688521.SH,688536.SH,688538.SH,688567.SH,688690.SH,688772.SH,688777.SH,688779.SH,688819.SH,689009.SH
20040102,0,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
20040105,0,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
20040106,0,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
20040107,0,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
20040108,0,1,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20230118,0,0,0,1,1,0,1,0,1,0,...,1,0,1,1,1,1,1,1,1,1
20230119,0,0,0,1,1,0,1,0,1,0,...,1,0,1,1,1,1,1,1,1,1
20230120,0,0,0,1,1,0,1,0,1,0,...,1,0,1,1,1,1,1,1,1,1
20230130,0,0,0,1,1,0,1,0,1,0,...,1,0,1,1,1,1,1,1,1,1


In [8]:
closeprice

,000001.SH,000016.SH,000063.SH,000300.SH,000827.SH,000852.SH,000903.SH,000905.SH,000906.SH,000913.SH,...,HSI.HI,IBOVESPA.GI,IXIC.GI,KS11.GI,N225.GI,SENSEX.GI,SPX.GI,TWII.TW,USDX.FX,^W5000
19500103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,16.66000,NaN,NaN,NaN
19500104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,16.85000,NaN,NaN,NaN
19500105,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,16.93000,NaN,NaN,NaN
19500106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,16.98000,NaN,NaN,NaN
19500109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,17.08000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20230118,3224.4060,2810.0860,2950.8459,4130.3143,2244.6876,6613.0050,4018.3248,6151.4557,4464.3690,11646.1059,...,26343.099609,117857.0,12771.110352,2759.820068,26524.789062,46444.179688,3690.01001,14223.089844,102.4052,39091.019531
20230119,3240.2794,2820.8067,2964.9399,4156.0077,2234.3433,6676.0803,4046.8916,6204.3261,4494.8031,11902.4637,...,26343.099609,117857.0,12771.110352,2759.820068,26524.789062,46444.179688,3690.01001,14223.089844,102.0712,39091.019531
20230120,3264.8138,2836.8070,2986.4187,4181.5267,2279.2669,6736.8515,4079.3644,6251.4337,4524.0457,11849.8425,...,26343.099609,117857.0,12771.110352,2759.820068,26524.789062,46444.179688,3690.01001,14223.089844,101.9966,39091.019531
20230130,3269.3180,2839.1216,2978.3743,4201.3450,2313.6305,6786.4530,4100.8178,6283.3272,4545.9003,11749.1858,...,26343.099609,117857.0,12771.110352,2759.820068,26524.789062,46444.179688,3690.01001,14223.089844,102.2304,39091.019531


In [9]:
contents_000905 = contents_000905.loc['20181030':'20201030']

# 这里的每一天指的是交易日
every_day_dict_500 = {}

for i in range(0,  len(contents_000905)):

    # 找出对应的那天
    date = contents_000905.iloc[i].name

    stocks = [column for column in contents_000905.columns if contents_000905.iloc[i][column]]

    market_value_sum = 0

    PB_net_value_sum = 0

    PE_NET_PROFIT_parent = 0

    for stock in stocks:

        try:
            current_stock = grouped_df.get_group(stock).sort_values('TRADE_DT')
        except:
            continue

        try:
            date_index = current_stock[current_stock['TRADE_DT'] == date].index[0]
        except:
            continue

    
        S_VAL_MV = current_stock.loc[date_index, 'S_VAL_MV'] * 10000

        NET_ASSETS_TODAY = current_stock.loc[date_index, 'NET_ASSETS_TODAY']

        NET_PROFIT_PARENT_COMP_TTM = current_stock.loc[date_index, 'NET_PROFIT_PARENT_COMP_TTM']

        market_value_sum += S_VAL_MV

        PB_net_value_sum += NET_ASSETS_TODAY
        
        PE_NET_PROFIT_parent += NET_PROFIT_PARENT_COMP_TTM

    riskfree_ = riskfree.loc[riskfree['date'] == date, 'interest'].values[0]

    PB = market_value_sum / PB_net_value_sum

    PE = market_value_sum / PE_NET_PROFIT_parent

    PBPE = (PB * PE) ** 0.5

    RISK_PREMIERE = 1/PE - riskfree_

    every_day_dict_500[date] = [PB,PE,PBPE, RISK_PREMIERE]


In [10]:
# Convert the dictionary into a DataFrame
df_500 = pd.DataFrame.from_dict(every_day_dict_500, orient='index', columns=['PB', 'PE', 'PBPE', 'RISK_PREMIERE'])

In [11]:
df_500

,PB,PE,PBPE,RISK_PREMIERE
20181030,1.553238,17.577739,5.225170,-3.464010
20181031,1.577496,18.278784,5.369796,-3.463792
20181101,1.587840,18.398638,5.405006,-3.440648
20181102,1.638703,18.988664,5.578241,-3.469537
20181105,1.638643,18.987659,5.577993,-3.486134
...,...,...,...,...
20201026,2.083567,31.352610,8.082405,-3.192005
20201027,2.085930,31.400594,8.093173,-3.180053
20201028,2.088107,31.344075,8.090103,-3.162496
20201029,2.085904,31.225431,8.070517,-3.174675


In [12]:
closeprice_500

20181030    4204.5424
20181031    4272.5518
20181101    4298.9837
20181102    4437.9456
20181104    4437.9456
              ...    
20221123    6115.6049
20221124    6120.0659
20221125    6087.8591
20221128    6055.2843
20221129    6163.8214
Name: 000905.SH, Length: 1087, dtype: float64

In [13]:
df = df_500

dfcloseprice = pd.DataFrame(closeprice)
dfcloseprice = closeprice_500
# print(dfcloseprice)
# 这里我们可以理解为工作日
DAY = 50
XDAYLATER3 = 20

bottom_thresholds = [1, 3, 5, 8, 10]
top_thresholds = [99, 97, 95, 92, 90]
bottom_signals = {}
top_signals = {}
results = {}

for i in range(DAY, len(df)):
    # end is like 20221124
    end = df.iloc[i].name

    interval_data = df.loc[:end][-DAY:]

    today_pb = df.iloc[i]['PB']
    pb_percentile = percentileofscore(interval_data['PB'], today_pb)

    today_pe = df.iloc[i]['PE']
    pe_percentile = percentileofscore(interval_data['PE'], today_pe)

    today_pbpe = df.iloc[i]['PBPE']
    pbpe_percentile = percentileofscore(interval_data['PBPE'], today_pbpe)

    today_risk_premiere = df.iloc[i]['RISK_PREMIERE']
    risk_premiere_percentile = percentileofscore(interval_data['RISK_PREMIERE'], today_risk_premiere)

    index = dfcloseprice.index.get_loc(str(end)) + XDAYLATER3

    first_price = dfcloseprice.iloc[index-20]

    try:
        # Fetch the price XDAYLATER3 days later
        end_price = dfcloseprice.iloc[index + XDAYLATER3]
    except:
        break


    # Check if any percentiles exceed the top thresholds
    for threshold in top_thresholds:
        if end_price is None:
            state = 'None'
        elif first_price > end_price:
            state = 1
        else:
            state = 0
        

        if pb_percentile > threshold:
            top_signals.setdefault(threshold, []).append((end, threshold, pb_percentile, today_pb, "PB", state))
            # break
        if pe_percentile > threshold:
            top_signals.setdefault(threshold, []).append((end, threshold, pe_percentile, today_pe, "PE", state))
            # break
        if pbpe_percentile > threshold:
            top_signals.setdefault(threshold, []).append((end, threshold,pbpe_percentile, today_pbpe, "PBPE", state))
            # break
        if risk_premiere_percentile > threshold:
            top_signals.setdefault(threshold, []).append((end, threshold,risk_premiere_percentile, today_risk_premiere, "RISK_PREMIERE", state))


    # Check if any percentiles fall below the bottom thresholds
    for threshold in bottom_thresholds:
        if end_price is None:
            state = 'None'
        elif first_price < end_price:
            state = 1
        else:
            state = 0
        
        # 观察是否触发
        if pb_percentile < threshold:
            bottom_signals.setdefault(threshold, []).append((end, threshold,pb_percentile, today_pb, "PB",state))
            # break
        if pe_percentile < threshold:
            bottom_signals.setdefault(threshold, []).append((end, threshold,pe_percentile, today_pe, "PE",state))
            # break
        if pbpe_percentile < threshold:
            bottom_signals.setdefault(threshold, []).append((end, threshold,pbpe_percentile, today_pbpe, "PBPE",state))
            # break
        if risk_premiere_percentile < threshold:
            bottom_signals.setdefault(threshold, []).append((end, threshold,risk_premiere_percentile, today_risk_premiere, "RISK_PREMIERE", state))

    
combined_signals = {}
combined_signals.update(top_signals)
combined_signals.update(bottom_signals)

In [14]:
rows = []
for threshold, signals in combined_signals.items():
    for signal in signals:
        # try:
        row = [signal[0], signal[1], signal[2], signal[3], signal[4],signal[5]]
        # except:
        #     row = [signal[0], signal[1], signal[2], signal[3], signal[4]]
        rows.append(row)

# Define the header row
header = ['Date', 'Threshold', 'Percentile', 'Value', 'Factor','20daystate']
df_result = pd.DataFrame(rows,columns=header)


In [15]:
df_result

,Date,Threshold,Percentile,Value,Factor,20daystate
0,20190110,99,100.0,-3.048929,RISK_PREMIERE,0
1,20190111,99,100.0,-3.044560,RISK_PREMIERE,0
2,20190116,99,100.0,-3.040421,RISK_PREMIERE,0
3,20190117,99,100.0,-3.001369,RISK_PREMIERE,0
4,20190214,99,100.0,1.679125,PB,0
...,...,...,...,...,...,...
2096,20201029,8,6.0,31.225431,PE,1
2097,20201029,8,2.0,8.070517,PBPE,1
2098,20201030,8,2.0,2.030671,PB,1
2099,20201030,8,2.0,30.282798,PE,1


In [16]:

results = []

for group_name, group_data in df_result.groupby(['Threshold','Factor']):
    percentage3 = group_data['20daystate'].astype(bool).mean() * 100
    count = len(group_data)
    results.append({'indicator': group_name, 'winning ratio': percentage3,'time': count})

percentage_df = pd.DataFrame(results)
percentage_df = percentage_df.reset_index(drop=True)
percentage_df = percentage_df.sort_values(by= 'winning ratio', ascending= False)

In [17]:
percentage_df

,indicator,winning ratio,time
4,"(5, PB)",88.000000,25
0,"(3, PB)",86.666667,15
8,"(8, PB)",83.333333,36
12,"(10, PB)",82.608696,46
6,"(5, PE)",81.818182,11
1,"(3, PBPE)",81.250000,16
2,"(3, PE)",80.000000,10
5,"(5, PBPE)",78.260870,23
14,"(10, PE)",76.923077,26
9,"(8, PBPE)",76.666667,30


In [18]:
df_pd_nowadays = pd.read_excel('市场整体估值.xlsx',index_col=0)
df_pd_nowadays

,日期,中证500,中证500.1,中证500.2
序号,,,,
NaN,NaN,收盘,市盈率(TTM),市净率(LF)
1.0,2024-01-19 00:00:00,5018.4332,20.286512,1.5525
2.0,2024-01-12 00:00:00,5206.178,21.054677,1.611462
3.0,2024-01-05 00:00:00,5281.8711,21.358787,1.63503
4.0,2023-12-29 00:00:00,5429.2288,21.836943,1.67265
...,...,...,...,...
153.0,2021-01-22 00:00:00,6637.8052,29.426011,2.132942
154.0,2021-01-15 00:00:00,6417.5011,28.459617,2.065596
155.0,2021-01-08 00:00:00,6557.5964,29.050314,2.108869


In [19]:
np.percentile(df_pd_nowadays.iloc[1:-3,-1], 5)

1.6149401692835625

In [21]:
np.percentile(df_pd_nowadays.iloc[1:-3,-1], 10)

1.6351251921437284